<a href="https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/17_limpieza.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

🎬 [La escena en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cap17) · 🗺️ [El mapa](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html) · ⬅️ [Cap. 16](https://colab.research.google.com/github/farithjhg/aws-bedrock-agentcore-course/blob/main/v7_aprender_haciendo/labs/16_tu_escena.ipynb)

# Capítulo 17 — Apagarlo todo

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** El curso ha terminado. Nova Assist, NovaWeb, el equipo de investigación, los datos y los registros siguen ahí. Unos cobran solo si se usan; otros cobran por existir. ¿Sabes cuáles?

| | |
|---|---|
| **Objetivo** | Dejar la cuenta como estaba antes del curso, sin nada del laboratorio facturando |
| **Quién lo ejecuta** | Cada participante en su cuenta. El terreno (sección 8) lo apaga el administrador con `terraform destroy` |
| **Duración orientativa** | 20 min (la mayor parte, esperando a CloudFormation) |
| **Se puede repetir** | Sí. Cada paso comprueba si el recurso existe y continúa aunque algo falle; si un paso falla, corrige la causa y vuelve a ejecutar el notebook entero |

> ⚠️ Este capítulo **elimina recursos** de la cuenta: todo lo que encuentre del laboratorio (nombres con «Nova»), sea de quien sea. Por eso cada participante tiene su propia cuenta.

### ☁️ ¿Estás en Google Colab? Ejecuta primero esta celda

En tu equipo no hace nada. En Colab descarga el curso, monta tu **Google Drive** (ahí se guarda tu estado entre capítulos: `MyDrive/NovaAssist_curso`), instala las dependencias y carga tus credenciales de AWS desde los **Secretos** de Colab. Tarda 2–3 minutos, porque cada notebook abre una máquina nueva: mientras tanto, lee la escena.

In [ ]:
# ☁️ Solo en Google Colab (en tu equipo no hace nada)
import os, sys, subprocess
if "google.colab" in sys.modules:
    CURSO = "/content/aws-bedrock-agentcore-course"
    if not os.path.isdir(CURSO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "main",
                        "https://github.com/farithjhg/aws-bedrock-agentcore-course.git", CURSO], check=True)
    os.chdir(f"{CURSO}/v7_aprender_haciendo/labs")
    sys.path.insert(0, os.getcwd())
    import colab_nova
    colab_nova.preparar(capitulo=17)

In [ ]:
import os
import time
import boto3
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
r = region()
ctl = boto3.client("bedrock-agentcore-control", region_name=r)
dp = boto3.client("bedrock-agentcore", region_name=r)
s = state_load()

## 0. Inventario

> ✋ **Predice antes de ejecutar.** De todo lo creado en el curso, ¿qué desaparecería con `agentcore remove all` + `agentcore deploy` y qué no? Haz las dos listas antes de mirar.

In [ ]:
antes = inventario_laboratorio()

**Lo que gestiona la CLI** (runtimes, endpoints, memoria, gateways y targets, políticas, evaluaciones online, *configuration bundles*, *credential providers*) vive en **stacks de CloudFormation** (`AgentCore-NovaAssist-…`). **Todo lo demás** lo creamos con boto3 a lo largo del curso y la CLI no lo conoce: Lambdas y sus roles, pools de Cognito, el guardrail, el *rate limit*, el A/B test, los datos (S3, S3 Tables, S3 Vectors) y los grupos de logs. Eso hay que borrarlo a mano, y en orden.

Qué cobra **por existir** aunque nadie lo use: los datos (S3, S3 Tables, S3 Vectors, registros de Memory), los logs de CloudWatch y los *spans* indexados de X-Ray. Runtime, Gateway, Lambda o Code Interpreter solo cobran cuando se usan, pero conviene borrarlos igual: son puertas abiertas en tu cuenta.

In [ ]:
CONFIRMAR_LIMPIEZA = False   # ← cámbialo a True de forma consciente

## 1. Lo que está en marcha: el A/B test y el *rate limit*

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
for t in dp.list_ab_tests().get("abTests", []):
    if t["abTestId"] == s.get("AB_TEST_ID") or t["name"] == "tono_breve" or "nova" in t["name"].lower():
        if t.get("executionStatus") == "RUNNING":
            intentar(f"Detener A/B test {t['name']}", lambda t=t: dp.update_ab_test(abTestId=t["abTestId"], executionStatus="STOPPED"))
        intentar(f"Borrar A/B test {t['name']}", lambda t=t: dp.delete_ab_test(abTestId=t["abTestId"]))

for g in ctl.list_gateways().get("items", []):
    if "nova" in g["name"].lower():
        for rl in ctl.list_gateway_rate_limits(gatewayIdentifier=g["gatewayId"]).get("rateLimits", []):
            intentar(f"Rate limit {rl['rateLimitId']} de {g['name']}",
                     lambda g=g, rl=rl: ctl.delete_gateway_rate_limit(gatewayIdentifier=g["gatewayId"], rateLimitId=rl["rateLimitId"]))

## 2. Lo que gestiona la CLI: los stacks de CloudFormation

La CLI ofrece `agentcore remove all` seguido de `agentcore deploy`, que hace lo mismo a través de CDK. Aquí borramos los stacks directamente: funciona aunque hayas perdido la carpeta del proyecto, e incluye los stacks auxiliares que crea el A/B test.

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
cfn = boto3.client("cloudformation", region_name=r)
activos = ["CREATE_COMPLETE", "UPDATE_COMPLETE", "UPDATE_ROLLBACK_COMPLETE", "ROLLBACK_COMPLETE", "DELETE_FAILED",
           "CREATE_FAILED", "UPDATE_ROLLBACK_FAILED", "IMPORT_COMPLETE"]
stacks = [st["StackName"] for p in cfn.get_paginator("list_stacks").paginate(StackStatusFilter=activos)
          for st in p["StackSummaries"] if st["StackName"].startswith("AgentCore-NovaAssist")]
print("Stacks del laboratorio:", stacks or "ninguno")
for nombre in stacks:
    intentar(f"Borrado iniciado: {nombre}", lambda n=nombre: cfn.delete_stack(StackName=n))
for nombre in stacks:
    try:
        cfn.get_waiter("stack_delete_complete").wait(StackName=nombre, WaiterConfig={"Delay": 20, "MaxAttempts": 60})
        print("✔ Stack borrado:", nombre)
    except Exception as e:  # noqa: BLE001
        print("✗", nombre, "→", str(e)[:200])
        print("  Motivo:", [ev["ResourceStatusReason"] for ev in cfn.describe_stack_events(StackName=nombre)["StackEvents"]
                            if ev.get("ResourceStatus") == "DELETE_FAILED"][:3])

Si un stack queda en `DELETE_FAILED`, suele ser porque algo creado fuera de CloudFormation depende de él (por ejemplo, un *target* añadido desde la consola). El barrido siguiente lo elimina; después **vuelve a ejecutar la celda anterior**.

## 3. Barrido de AgentCore: lo que haya quedado fuera de los stacks

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
def es_nova(nombre):
    return "nova" in (nombre or "").lower()

for g in ctl.list_gateways().get("items", []):
    if es_nova(g["name"]):
        for t in ctl.list_gateway_targets(gatewayIdentifier=g["gatewayId"]).get("items", []):
            intentar(f"Target {t['name']} de {g['name']}",
                     lambda g=g, t=t: ctl.delete_gateway_target(gatewayIdentifier=g["gatewayId"], targetId=t["targetId"]))
        time.sleep(5)
        intentar(f"Gateway {g['name']}", lambda g=g: ctl.delete_gateway(gatewayIdentifier=g["gatewayId"]))
for rt in ctl.list_agent_runtimes().get("agentRuntimes", []):
    if es_nova(rt["agentRuntimeName"]):
        intentar(f"Runtime {rt['agentRuntimeName']}", lambda rt=rt: ctl.delete_agent_runtime(agentRuntimeId=rt["agentRuntimeId"]))
for m in ctl.list_memories().get("memories", []):
    if es_nova(m.get("id")):
        intentar(f"Memoria {m['id']}", lambda m=m: ctl.delete_memory(memoryId=m["id"]))
for pe in ctl.list_policy_engines().get("policyEngines", []):
    if es_nova(pe["name"]):
        for p in ctl.list_policies(policyEngineId=pe["policyEngineId"]).get("policies", []):
            intentar(f"Política {p['name']}", lambda pe=pe, p=p: ctl.delete_policy(policyEngineId=pe["policyEngineId"], policyId=p["policyId"]))
        time.sleep(5)
        intentar(f"Motor de políticas {pe['name']}", lambda pe=pe: ctl.delete_policy_engine(policyEngineId=pe["policyEngineId"]))
for oe in ctl.list_online_evaluation_configs().get("onlineEvaluationConfigs", []):
    if es_nova(oe["onlineEvaluationConfigName"]):
        intentar(f"Evaluación online {oe['onlineEvaluationConfigName']}",
                 lambda oe=oe: ctl.delete_online_evaluation_config(onlineEvaluationConfigId=oe["onlineEvaluationConfigId"]))
for b in ctl.list_configuration_bundles().get("bundles", []):
    if es_nova(b["bundleName"]):
        intentar(f"Configuration bundle {b['bundleName']}", lambda b=b: ctl.delete_configuration_bundle(bundleId=b["bundleId"]))
for cp in ctl.list_oauth2_credential_providers().get("credentialProviders", []):
    if es_nova(cp["name"]):
        intentar(f"Credential provider {cp['name']}", lambda cp=cp: ctl.delete_oauth2_credential_provider(name=cp["name"]))
for cp in ctl.list_api_key_credential_providers().get("credentialProviders", []):
    if es_nova(cp["name"]):
        intentar(f"API key provider {cp['name']}", lambda cp=cp: ctl.delete_api_key_credential_provider(name=cp["name"]))
for wi in ctl.list_workload_identities().get("workloadIdentities", []):
    if es_nova(wi["name"]):
        intentar(f"Workload identity {wi['name']}", lambda wi=wi: ctl.delete_workload_identity(name=wi["name"]))

## 4. Lambdas y sus roles

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
lam = boto3.client("lambda", region_name=r)
for funcion, rol in (("nova-pedidos", "nova-pedidos-role"), ("nova-politicas", "nova-politicas-role")):
    intentar(f"Lambda {funcion}", lambda f=funcion: lam.delete_function(FunctionName=f))
    intentar(f"Rol {rol}", lambda r_=rol: borrar_rol_iam(r_))

## 5. Identidad y protección: Cognito y el guardrail

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
idp = boto3.client("cognito-idp", region_name=r)
for pool in idp.list_user_pools(MaxResults=60)["UserPools"]:
    if pool["Name"] in ("nova-m2m", "nova-clientes"):
        dominio = idp.describe_user_pool(UserPoolId=pool["Id"])["UserPool"].get("Domain")
        if dominio:
            intentar(f"Dominio {dominio}", lambda p=pool, d=dominio: idp.delete_user_pool_domain(Domain=d, UserPoolId=p["Id"]))
        intentar(f"Pool {pool['Name']}", lambda p=pool: idp.delete_user_pool(UserPoolId=p["Id"]))
bedrock = boto3.client("bedrock", region_name=r)
for g in bedrock.list_guardrails()["guardrails"]:
    if g["name"] == "nova-assist-guardrail":
        intentar("Guardrail nova-assist-guardrail", lambda g=g: bedrock.delete_guardrail(guardrailIdentifier=g["id"]))

## 6. Los datos: S3 Vectors, las tablas Iceberg y lo que hay en el bucket del laboratorio

El *table bucket*, el namespace y el bucket del laboratorio son del terreno (Terraform): aquí se borra lo que los notebooks pusieron dentro.

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
s3v = boto3.client("s3vectors", region_name=r)
for vb in s3v.list_vector_buckets().get("vectorBuckets", []):
    if vb["vectorBucketName"].startswith("nova-"):
        for ix in s3v.list_indexes(vectorBucketName=vb["vectorBucketName"]).get("indexes", []):
            intentar(f"Índice {ix['indexName']}", lambda vb=vb, ix=ix: s3v.delete_index(vectorBucketName=vb["vectorBucketName"], indexName=ix["indexName"]))
        intentar(f"Vector bucket {vb['vectorBucketName']}", lambda vb=vb: s3v.delete_vector_bucket(vectorBucketName=vb["vectorBucketName"]))

# Las tablas Iceberg que cargaron los notebooks (el table bucket y el namespace son del terreno: los borra Terraform)
s3t = boto3.client("s3tables", region_name=r)
if s.get("TABLE_BUCKET_ARN"):
    for t in s3t.list_tables(tableBucketARN=s["TABLE_BUCKET_ARN"]).get("tables", []):
        intentar(f"Tabla {t['namespace'][0]}.{t['name']}", lambda t=t: s3t.delete_table(tableBucketARN=s["TABLE_BUCKET_ARN"], namespace=t["namespace"][0], name=t["name"]))

# Lo que los notebooks dejaron en el bucket del laboratorio (el bucket es del terreno)
if s.get("LAB_BUCKET"):
    s3 = boto3.client("s3", region_name=r)
    def _vaciar_lab():
        for pagina in s3.get_paginator("list_object_versions").paginate(Bucket=s["LAB_BUCKET"]):
            objetos = [{"Key": o["Key"], "VersionId": o["VersionId"]} for k in ("Versions", "DeleteMarkers") for o in pagina.get(k, [])]
            if objetos:
                s3.delete_objects(Bucket=s["LAB_BUCKET"], Delete={"Objects": objetos, "Quiet": True})
    intentar(f"Objetos de {s['LAB_BUCKET']}", _vaciar_lab)

## 7. Los logs

Los grupos de logs del laboratorio no caducan nunca por defecto: cobran almacenamiento cada mes.

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
logs = boto3.client("logs", region_name=r)
for grupo in grupos_de_logs_laboratorio():
    intentar(f"Logs {grupo}", lambda g=grupo: logs.delete_log_group(logGroupName=g))

## 8. El terreno de la cuenta (solo el administrador)

Lo que creó Terraform en la Parte A del capítulo 0 se apaga con Terraform, **solo si la cuenta deja de usarse para la formación**:

```bash
cd infra/cuenta
terraform destroy          # varias cuentas: -state=estados/<perfil>.tfstate, como en el apply
```

Borra el presupuesto, desactiva Transaction Search (el destino de las trazas vuelve a X-Ray y se retira la política), borra el catálogo `s3tablescatalog` y los dos buckets con todo lo que tengan. Si la cuenta se reutiliza en otra edición, no lo destruyas: la Parte C del capítulo 0 recarga los datos. Recomendación: **conserva el presupuesto un mes** (`terraform destroy -target=module.cuenta.aws_s3tables_table_bucket.datos` y compañía, o simplemente espera) para ver los costes residuales.

> **Comprueba en Cost Explorer al día siguiente** que no hay cargos nuevos: es la verdadera comprobación final.

## 9. Comprobación final

> ✋ **Predice antes de ejecutar.** ¿Quedará algo? Si queda, ¿qué crees que será?

In [ ]:
despues = inventario_laboratorio()
if despues:
    print("\nQueda algo: revisa los mensajes de arriba, corrige la causa y vuelve a ejecutar el notebook (es idempotente).")

Algunos borrados son asíncronos (runtimes, gateways, memoria): si aparecen en la lista con pocos minutos de diferencia, espera y vuelve a ejecutar la comprobación.

## 10. Estado local

`lab_state.json` y la carpeta `partidas/` contienen secretos de laboratorio (el *secret* del cliente M2M, la contraseña de los usuarios de prueba). Los recursos ya no existen, pero no hay motivo para conservarlos.

In [ ]:
assert CONFIRMAR_LIMPIEZA, "Limpieza no confirmada"
import shutil
archivar_estado()
if PARTIDAS.exists():
    shutil.rmtree(PARTIDAS)
    print("Carpeta partidas/ borrada")
env_local = PROJECT / "agentcore" / ".env.local"
if env_local.exists():
    env_local.unlink()
    print("NovaAssist/agentcore/.env.local borrado")

## 🔎 Lo que acabamos de descubrir

Un sistema agéntico no es un programa: es un **conjunto de servicios** con datos, identidades, permisos y registros repartidos por la cuenta. Lo que declaras como código (la CLI y sus stacks) se borra de una vez; lo demás, solo si lo inventariaste. Esa es la última lección del curso: **si no sabes qué has creado, no sabes qué estás pagando**.

🎓 **Fin del laboratorio.**

---
**Fin del curso.** 🎬 [El cierre en la presentación](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/presentacion_nova_assist.html#cierre) · 🗺️ [El mapa del agente](https://farithjhg.github.io/aws-bedrock-agentcore-course/v7_aprender_haciendo/labs/teoria/index.html)